# 04 · World model (Part B, GPU runtime)

An action-conditioned world model of the agentview camera (`p2p/wm/`): from **one real frame** and a sequence of robot actions it imagines the next seconds, and a head reads the bowl position from every imagined frame.

Data:
- **train / val**: the generated episodes of notebook 02 (successes *and* failures, 10 training demos), plus a few episodes of the two other retargeting strategies, so the model has seen their kinds of motion.
- **test**: new episodes of the **3 held-out human demos** (never used for generation or training), each replayed with all three strategies in the *same* fresh scenes (paired seeds).

Questions (all on the test episodes):
1. image prediction vs horizon, against "copy the start frame";
2. bowl-position error vs horizon, against "the bowl stays put";
3. can it tell success from failure by imagining the end of an episode (AUROC vs how far it imagines), and does it rank the three strategies like the simulator does?

Runtime on a T4: ~45 min generation (resumable), ~20 min training, ~5 min evaluation.

In [ ]:
import os, sys, json, subprocess
os.environ["MUJOCO_GL"] = "egl"                      # GPU rendering, must be set before mujoco is imported
REPO_URL = "https://github.com/LukasWill/phone2panda.git"
BASE = "/content" if os.path.exists("/content") else "/kaggle/working"   # Colab or Kaggle
CODE_DIR = f"{BASE}/phone2panda"
if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lerobot[libero]==0.6.1", "imageio-ffmpeg"], check=True)
# Colab preinstalls an older torchao; a newer diffusers (pulled in by lerobot) then fails on import
# (FqnToConfig). ACT does not use torchao, and diffusers skips it when it is absent.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
from p2p.sim.env import ensure_nvidia_egl
ensure_nvidia_egl()      # Colab: make EGL render on the GPU (otherwise it silently renders on the CPU, ~10x slower)
# Hugging Face token (Colab: key icon -> Secrets -> HF_TOKEN; Kaggle: Add-ons -> Secrets -> HF_TOKEN)
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN found: uploads to the Hub will fail")
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import whoami
    print("Hugging Face account:", whoami(token=os.environ["HF_TOKEN"])["name"])
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
HUB_REPO = "phone2panda-data"   # same private dataset repo as notebook 02
HOLDOUT_PER_DEMO = 20           # test: 3 held-out demos x 3 strategies x 20 (same 60 scenes for every strategy)
EXTRA_PER_DEMO = 8              # train: 10 demos x {absolute, intent} x 8, so their motions are not out of distribution
STEPS, HORIZON = 20000, 32      # training steps; open-loop training horizon (10 Hz steps)
WORKERS = 2
HOLDOUT = ",".join(json.load(open("data/splits.json"))["holdout"])
OUT = f"{BASE}/wm"

## Data: notebook 02's episodes from the Hub, plus test and extra episodes

In [ ]:
from p2p.hub import pull
if not os.path.exists(f"{BASE}/generated"):
    pull(HUB_REPO, f"{BASE}/hub", allow_patterns=["raw/generated.tar"])
    !tar -xf {BASE}/hub/raw/generated.tar -C {BASE}
print(len(os.listdir(f"{BASE}/generated")), "files from notebook 02")

In [ ]:
# test episodes: held-out demos only (--holdout "" lifts the exclusion, --demos selects them); same seeds for every strategy
for s in ("object_relative", "intent", "absolute"):
    !python -m p2p.data.generate data/processed {OUT}/test_{s} --strategy {s} --holdout "" --demos {HOLDOUT} --per-demo {HOLDOUT_PER_DEMO} --seed0 500000 --image-size 64 --workers {WORKERS}
# extra training episodes of the other two strategies (training demos only: the default holdout applies)
for s in ("intent", "absolute"):
    !python -m p2p.data.generate data/processed {OUT}/extra_{s} --strategy {s} --per-demo {EXTRA_PER_DEMO} --seed0 300000 --image-size 64 --workers {WORKERS}

In [ ]:
srcs = f"{BASE}/generated:gen " + " ".join(f"{OUT}/extra_{s}:extra_{s}" for s in ("intent", "absolute")) + " " + \
       " ".join(f"{OUT}/test_{s}:test_{s}" for s in ("object_relative", "intent", "absolute"))
!python -m p2p.wm.data {OUT}/cache.npz {srcs}

## Train

In [ ]:
!python -m p2p.wm.train {OUT}/cache.npz {OUT}/run --steps {STEPS} --horizon {HORIZON}

## Evaluate on the held-out demos

In [ ]:
!python -m p2p.wm.evaluate {OUT}/run/wm.pt {OUT}/cache.npz {OUT}/eval
from IPython.display import Image, Video, display
import pandas as pd
display(Image(f"{OUT}/eval/wm_horizon.png")); display(Image(f"{OUT}/eval/wm_auroc.png"))
display(pd.read_csv(f"{OUT}/eval/wm_strategies.csv"))
display(Image(f"{OUT}/eval/wm_hero_success_filmstrip.png")); display(Image(f"{OUT}/eval/wm_hero_failure_filmstrip.png"))
Video(f"{OUT}/eval/wm_hero_success.mp4", embed=True)

## Save: checkpoint and results to the Hub, results as a zip to commit into `results/wm/`

In [ ]:
from p2p.hub import push
push(f"{OUT}/run", HUB_REPO, path_in_repo="wm/run")
push(f"{OUT}/eval", HUB_REPO, path_in_repo="wm/eval")
!cd {OUT} && zip -qr wm_eval.zip eval run/train_log.json
try:
    from google.colab import files; files.download(f"{OUT}/wm_eval.zip")
except Exception:
    print("download", f"{OUT}/wm_eval.zip", "from the Output panel")